# Week 3: Feature Selection and Dimensionality Reduction (Forward/Backward Selection, PCR, PLSR)

## 1. Notebook Setup
- Import libraries  
- Define helper functions for model evaluation and feature selection

In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
import statsmodels.api as sm

#evaluate regression model
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return {"MSE": mse, "R2": r2}

# Forward selection
def forward_selection(X, y, significance_level=0.05):
    initial_features = []
    best_features = []
    while True:
        remaining_features = list(set(X.columns) - set(initial_features))
        new_pval = pd.Series(index=remaining_features)
        for new_column in remaining_features:
            model = sm.OLS(y, sm.add_constant(X[initial_features + [new_column]])).fit()
            new_pval[new_column] = model.pvalues[new_column]
        min_p_value = new_pval.min()
        if min_p_value < significance_level:
            best_feature = new_pval.idxmin()
            initial_features.append(best_feature)
            best_features.append(best_feature)
        else:
            break
    return best_features

# Backward elimination
def backward_elimination(X, y, significance_level=0.05):
    features = X.columns.tolist()
    while len(features) > 0:
        model = sm.OLS(y, sm.add_constant(X[features])).fit()
        max_p_value = model.pvalues.drop("const").max()
        if max_p_value >= significance_level:
            excluded_feature = model.pvalues.drop("const").idxmax()
            features.remove(excluded_feature)
        else:
            break
    return features

## 2. Dataset 1: customer_churn
### 2.1 Data Overview & Preparation

In [2]:
#load data
train_cc = pd.read_csv('customer_churn_dataset-training-master.csv')
test_cc = pd.read_csv('customer_churn_dataset-testing-master.csv')
#combine train and test set
customer_churn = pd.concat([train_cc, test_cc], ignore_index=True)
#drop null rows
customer_churn.dropna(inplace=True)
customer_churn.head()
customer_churn.info()
customer_churn.describe()

X = pd.get_dummies(customer_churn.drop("Churn", axis=1, errors="ignore"), drop_first=True)
y = customer_churn["Churn"] if "Churn" in customer_churn.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<class 'pandas.core.frame.DataFrame'>
Index: 505206 entries, 0 to 505206
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   CustomerID         505206 non-null  float64
 1   Age                505206 non-null  float64
 2   Gender             505206 non-null  object 
 3   Tenure             505206 non-null  float64
 4   Usage Frequency    505206 non-null  float64
 5   Support Calls      505206 non-null  float64
 6   Payment Delay      505206 non-null  float64
 7   Subscription Type  505206 non-null  object 
 8   Contract Length    505206 non-null  object 
 9   Total Spend        505206 non-null  float64
 10  Last Interaction   505206 non-null  float64
 11  Churn              505206 non-null  float64
dtypes: float64(9), object(3)
memory usage: 50.1+ MB


In [3]:
X = pd.get_dummies(customer_churn, columns=['Gender', 'Subscription Type', 'Contract Length'])

In [4]:
print(X)

        CustomerID   Age  Tenure  Usage Frequency  Support Calls  \
0              2.0  30.0    39.0             14.0            5.0   
1              3.0  65.0    49.0              1.0           10.0   
2              4.0  55.0    14.0              4.0            6.0   
3              5.0  58.0    38.0             21.0            7.0   
4              6.0  23.0    32.0             20.0            5.0   
...            ...   ...     ...              ...            ...   
505202     64370.0  45.0    33.0             12.0            6.0   
505203     64371.0  37.0     6.0              1.0            5.0   
505204     64372.0  25.0    39.0             14.0            8.0   
505205     64373.0  50.0    18.0             19.0            7.0   
505206     64374.0  52.0    45.0             15.0            9.0   

        Payment Delay  Total Spend  Last Interaction  Churn  Gender_Female  \
0                18.0        932.0              17.0    1.0           True   
1                 8.0      

In [5]:
print(y)

0         1.0
1         1.0
2         1.0
3         1.0
4         1.0
         ... 
505202    1.0
505203    1.0
505204    1.0
505205    1.0
505206    1.0
Name: Churn, Length: 505206, dtype: float64


### 2.2 Forward Selection

In [6]:
forward_selection(X_train, y_train)

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

In [7]:
if y is not None:
    selected_features_forward = forward_selection(X_train, y_train)
    print("Selected features (Forward):", selected_features_forward)

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 2.3 Backward Elimination

In [ ]:
if y is not None:
    selected_features_backward = backward_elimination(X_train, y_train)
    print("Selected features (Backward):", selected_features_backward)

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 2.4 Principal Component Regression (PCR)

In [8]:
if y is not None:
    pca = PCA(n_components=min(10, X_train.shape[1]))
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    lr = LinearRegression()
    results_pcr = evaluate_model(lr, X_train_pca, X_test_pca, y_train, y_test)
    print("PCR Results:", results_pcr)

PCR Results: {'MSE': 0.11638955931940605, 'R2': 0.528775410093898}


### 2.5 Partial Least Squares Regression (PLSR)

In [9]:
if y is not None:
    plsr = PLSRegression(n_components=min(10, X_train.shape[1]))
    plsr.fit(X_train, y_train)
    y_pred = plsr.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print("PLSR Results:", {"MSE": mse, "R2": r2})

PLSR Results: {'MSE': 0.11349207506182366, 'R2': 0.5405064093263264}


### 2.6 Insights
- Key findings for `customer_churn` go here

## 3. Dataset 2: digital_marketing_campaign
### 3.1 Data Overview & Preparation

In [10]:
digital_marketing = pd.read_csv("digital_marketing_campaign_dataset.csv")
digital_marketing.dropna(inplace=True)
X = pd.get_dummies(digital_marketing.drop("Conversion", axis=1, errors="ignore"), drop_first=True)
y = digital_marketing["Conversion"] if "Conversion" in digital_marketing.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### 3.2 Forward Selection

In [11]:
if y is not None:
    selected_features_forward = forward_selection(X_train, y_train)
    print("Selected features (Forward):", selected_features_forward)

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 3.3 Backward Elimination

In [ ]:

if y is not None:
    selected_features_backward = backward_elimination(X_train, y_train)
    print("Selected features (Backward):", selected_features_backward)


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 3.4 Principal Component Regression (PCR)

In [12]:

if y is not None:
    pca = PCA(n_components=min(10, X_train.shape[1]))
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    lr = LinearRegression()
    results_pcr = evaluate_model(lr, X_train_pca, X_test_pca, y_train, y_test)
    print("PCR Results:", results_pcr)


PCR Results: {'MSE': 0.09730716581309203, 'R2': 0.08673305684945376}


### 3.5 Partial Least Squares Regression (PLSR)

In [13]:

if y is not None:
    plsr = PLSRegression(n_components=min(10, X_train.shape[1]))
    plsr.fit(X_train, y_train)
    y_pred = plsr.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print("PLSR Results:", {"MSE": mse, "R2": r2})


PLSR Results: {'MSE': 0.0916124808601833, 'R2': 0.1401799687566203}


### 3.6 Insights
- Key findings for `digital_marketing_campaign` go here

## 4. Dataset 3: marketing_campaign (Optional for Week 3)
### 4.1 Data Overview & Preparation

In [14]:
marketing_campaign = pd.read_csv("marketing_campaign.csv", sep=';')
marketing_campaign.dropna(inplace=True)
X = pd.get_dummies(marketing_campaign.drop("Response", axis=1, errors="ignore"), drop_first=True)
y = marketing_campaign["Response"] if "Response" in marketing_campaign.columns else None

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


### 4.2 Forward Selection

In [15]:

if y is not None:
    selected_features_forward = forward_selection(X_train, y_train)
    print("Selected features (Forward):", selected_features_forward)


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 4.3 Backward Elimination

In [16]:

if y is not None:
    selected_features_backward = backward_elimination(X_train, y_train)
    print("Selected features (Backward):", selected_features_backward)


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

### 4.4 Principal Component Regression (PCR)

In [17]:

if y is not None:
    pca = PCA(n_components=min(10, X_train.shape[1]))
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)

    lr = LinearRegression()
    results_pcr = evaluate_model(lr, X_train_pca, X_test_pca, y_train, y_test)
    print("PCR Results:", results_pcr)


PCR Results: {'MSE': 0.10175302577681411, 'R2': 0.15304912643396262}


### 4.5 Partial Least Squares Regression (PLSR)

In [18]:

if y is not None:
    plsr = PLSRegression(n_components=min(10, X_train.shape[1]))
    plsr.fit(X_train, y_train)
    y_pred = plsr.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print("PLSR Results:", {"MSE": mse, "R2": r2})


PLSR Results: {'MSE': 0.13555247818151, 'R2': -0.12828379238262788}


### 4.6 Insights
- Key findings for `marketing_campaign` go here

## 5. Comparative Insights
- Compare results across datasets (Forward vs Backward vs PCR vs PLSR)
- Which feature selection worked best?
- Did dimensionality reduction improve performance?
- Takeaways for future modeling